### Import dependancies

In [1]:
import numpy as np
import cv2
from pathlib import Path
from collections import deque

### Set configurations

In [1]:
THRESHOLD = 8
MIN_AREA = 150

MAX_DISAPPEARANCE = 15
MAX_DISTANCE_THRESHOLD = 150

### Define object detection function

In [3]:
def merge_boxes(boxes, merge_distance=50):
    merged = []
    while boxes:
        x, y, w, h = boxes.pop(0)
        merged_any = False
        for i, (mx, my, mw, mh) in enumerate(merged):
            center_dist = np.hypot((x + w//2) - (mx + mw//2),(y + h//2) - (my + mh//2))
            if center_dist < merge_distance:
                nx = min(x, mx)
                ny = min(y, my)
                nw = max(x+w, mx+mw) - nx
                nh = max(y+h, my+mh) - ny
                merged[i] = (nx, ny, nw, nh)
                merged_any = True
                break
        if not merged_any:
            merged.append((x,y,w,h))

    return merged

In [4]:
def overlap_ratio(boxA, boxB):
    ax, ay, aw, ah = boxA
    bx, by, bw, bh = boxB
    ax2 = ax + aw
    ay2 = ay + ah
    bx2 = bx + bw
    by2 = by + bh
    inter_x1 = max(ax, bx)
    inter_y1 = max(ay, by)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    intersection = inter_w * inter_h
    smaller_area = min(aw * ah,bw * bh)
    if smaller_area == 0:
        return 0
    return intersection / smaller_area

In [35]:
def process_video(video_path, output_folder):
    video_path = Path(video_path)
    cap = cv2.VideoCapture(str(video_path))

    # Create subfolder for this video
    video_output_folder = (Path(output_folder) / video_path.stem)
    video_output_folder.mkdir(parents=True, exist_ok=True)

    tracked_objects = {}
    next_object_id = 0
    frame_index = 0
    SAVE_AFTER_FRAMES = 1

    print(f"\nProcessing: {video_path.name}")
    
    frame_buffer = deque(maxlen=3)

    '''ret, first_frame = cap.read()
    height, width = first_frame.shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    Path('thresh_videos').mkdir(exist_ok=True)
    thresh_video_path = Path('thresh_videos') / f"{video_path.stem}_thresh.mp4"
    print(thresh_video_path)
    thresh_writer = cv2.VideoWriter(thresh_video_path, fourcc, 30, (width, height), isColor=False)'''

    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))

    # read frames
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = clahe.apply(gray)
        gray = cv2.GaussianBlur(gray, (5,5), 0)

        # threshold 
        frame_buffer.append(gray)
        accumulated = gray.copy()
        for old_frame in frame_buffer:
            accumulated = cv2.max(accumulated, old_frame)

        _, thresh = cv2.threshold(accumulated, THRESHOLD , 255, cv2.THRESH_BINARY)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5))
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
        #thresh_writer.write(thresh)
        #cv2.imshow('thresh', thresh)
        #cv2.waitKey(1)

        # find contours
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        debug_folder = Path("debug")
        debug_frame = frame.copy()
        
        for contour in contours:
            area = cv2.contourArea(contour)

            x, y, w, h = cv2.boundingRect(contour)

            cv2.rectangle(debug_frame, (x,y), (x+w,y+h), (0,0,255), 1)
        save_path = debug_folder / f"debug_{frame_index}.jpg"
        if len(contours) > 0:
            cv2.imwrite(str(debug_folder / f"debug_{frame_index}.jpg"), debug_frame)
        print(save_path)
        '''current_frame_detections = []
        raw_boxes = []
        for contour in contours:
            area = cv2.contourArea(contour)
            if area < MIN_AREA:
                continue
            x, y, w, h = cv2.boundingRect(contour)
            debug_frame = frame.copy()
            cv2.rectangle(debug_frame, (x,y), (x+w, y+h), (0,0,255), 1)
            
            roi = gray[y:y+h, x:x+w]
            mean_brightness = np.mean(roi)
            if mean_brightness < 30:
                continue
                
            raw_boxes.append((x,y,w,h))
        
        merged_boxes = merge_boxes(raw_boxes)
        for x,y,w,h in merged_boxes:
            cx = x + w // 2
            cy = y + h // 2
            current_frame_detections.append((cx, cy, x, y, w, h))
        # track detected objects 
        updated_tracked_objects = {}
        matched_existing_ids = set()        # prevent duplicates 

        for cx, cy, x, y, w, h in current_frame_detections:
            matched_id = None
            min_dist = float("inf")
            for obj_id, (ox,oy,ow,oh,disappear_count,age,saved) in tracked_objects.items():
                # Prevent same ID matching twice
                if obj_id in matched_existing_ids:
                    continue

                ocx = ox + ow // 2
                ocy = oy + oh // 2
                old_area = ow * oh
                new_area = w*h
                size_ratio = min(old_area, new_area) / max(old_area, new_area)
                distance = np.hypot(cx - ocx, cy - ocy)
                current_box = (x, y, w, h)
                previous_box = (ox, oy, ow, oh)
                overlap = overlap_ratio(current_box, previous_box)
                # MATCH CONDITION
                if (overlap > 0.2 or distance < MAX_DISTANCE_THRESHOLD): # and size_ratio > 0.15:
                    if distance < min_dist:
                        min_dist = distance
                        matched_id = obj_id

            # if object exists
            if matched_id is not None:
                _, _, _, _, _, age, saved = tracked_objects[matched_id]
                updated_tracked_objects[matched_id] = ( x,y,w,h,0, age+1,saved)
                matched_existing_ids.add(matched_id)
                if (age + 1 >= SAVE_AFTER_FRAMES and not saved):
                    output_frame = frame.copy()
                    cv2.rectangle(output_frame,(x, y),(x + w, y + h),(0,255,0),2)
                    image_filename = (f"{video_path.stem}" f"_frame{frame_index}.jpg")
                    save_path = (video_output_folder / image_filename)
                    cv2.imwrite(str(save_path),output_frame)
                    print(f"[SAVED] {save_path}")
                    updated_tracked_objects[matched_id] = (x,y,w,h,0,age+1, True)
            # if new object 
            else:
                updated_tracked_objects[next_object_id] = (x,y,w,h,0,1,False)
                next_object_id += 1
        # count disappearances to avoid duplicates
        for obj_id, data in tracked_objects.items():
            if obj_id not in updated_tracked_objects:
                x, y, w, h, disappear_count, age, saved = data
                if disappear_count < MAX_DISAPPEARANCE:
                    updated_tracked_objects[obj_id] = (x,y,w,h,disappear_count+1, age, saved)
        tracked_objects = updated_tracked_objects'''
        frame_index += 1

    cap.release()
    print(f"Finished: {video_path.name}")

### Build path

In [6]:
def process_video_folder(input_folder, output_folder="detected_objects"):
    input_folder = Path(input_folder)
    video_files = [file for file in input_folder.iterdir()]

    print(f"Found {len(video_files)} videos")

    for video_file in video_files:
        process_video(video_file, output_folder)


In [31]:
if __name__ == "__main__":
    process_video_folder("test_converted_videos")

Found 6 videos

Processing: 10.0.16.2_202307070748.mp4
thresh_videos/10.0.16.2_202307070748_thresh.mp4
Finished: 10.0.16.2_202307070748.mp4

Processing: 10.0.16.2_202309020548.mp4
thresh_videos/10.0.16.2_202309020548_thresh.mp4
Finished: 10.0.16.2_202309020548.mp4

Processing: 10.0.12.2_202309020548.mp4
thresh_videos/10.0.12.2_202309020548_thresh.mp4
Finished: 10.0.12.2_202309020548.mp4

Processing: 10.0.11.2_202307070748.mp4
thresh_videos/10.0.11.2_202307070748_thresh.mp4
Finished: 10.0.11.2_202307070748.mp4

Processing: 10.0.12.2_202307070748.mp4
thresh_videos/10.0.12.2_202307070748_thresh.mp4
Finished: 10.0.12.2_202307070748.mp4

Processing: 10.0.11.2_202309020548.mp4
thresh_videos/10.0.11.2_202309020548_thresh.mp4
Finished: 10.0.11.2_202309020548.mp4


In [37]:
process_video("10.0.12.2_202307070748.mp4",'debug')


Processing: 10.0.12.2_202307070748.mp4
debug/debug_0.jpg
debug/debug_1.jpg
debug/debug_2.jpg
debug/debug_3.jpg
debug/debug_4.jpg
debug/debug_5.jpg
debug/debug_6.jpg
debug/debug_7.jpg
debug/debug_8.jpg
debug/debug_9.jpg
debug/debug_10.jpg
debug/debug_11.jpg
debug/debug_12.jpg
debug/debug_13.jpg
debug/debug_14.jpg
debug/debug_15.jpg
debug/debug_16.jpg
debug/debug_17.jpg
debug/debug_18.jpg
debug/debug_19.jpg
debug/debug_20.jpg
debug/debug_21.jpg
debug/debug_22.jpg
debug/debug_23.jpg
debug/debug_24.jpg
debug/debug_25.jpg
debug/debug_26.jpg
debug/debug_27.jpg
debug/debug_28.jpg
debug/debug_29.jpg
debug/debug_30.jpg
debug/debug_31.jpg
debug/debug_32.jpg
debug/debug_33.jpg
debug/debug_34.jpg
debug/debug_35.jpg
debug/debug_36.jpg
debug/debug_37.jpg
debug/debug_38.jpg
debug/debug_39.jpg
debug/debug_40.jpg
debug/debug_41.jpg
debug/debug_42.jpg
debug/debug_43.jpg
debug/debug_44.jpg
debug/debug_45.jpg
debug/debug_46.jpg
debug/debug_47.jpg
debug/debug_48.jpg
debug/debug_49.jpg
debug/debug_50.jpg
d